# 📘 Módulo 9 — Árvores de Decisão e Random Forest
## Modelos Não Lineares, Interpretabilidade e Ensambles

- 🔵 Conteúdo essencial (IBM‑like)
- 🟣 Conteúdo expandido (Livro Didático)
- 🟠 Conteúdo avançado (opcional)

Este módulo introduz dois dos modelos mais importantes do Machine Learning moderno:

# **Árvores de Decisão**  
# **Random Forest**

Eles são:
- não lineares  
- interpretáveis  
- robustos  
- capazes de capturar interações automaticamente  
- base de modelos mais avançados (XGBoost, LightGBM, CatBoost)  

---

# 🎯 Objetivos do Módulo 9

Ao final deste módulo, você será capaz de:

- entender como árvores de decisão funcionam  
- interpretar divisões, nós e folhas  
- ajustar árvores para classificação e regressão  
- controlar profundidade, overfitting e complexidade  
- visualizar árvores  
- entender o conceito de ensamble  
- ajustar Random Forest  
- interpretar importância de variáveis  
- comparar árvores vs regressão logística  
- construir um pipeline completo com Random Forest  

Este módulo é essencial para dominar ML moderno.

---

# 📚 Índice

0. [Setup — bibliotecas e dados](#setup)
1. [Introdução: por que árvores?](#intro)
2. [Árvores de Decisão — Intuição](#intuicao)
3. [Árvores de Classificação](#classificacao)
4. [Árvores de Regressão](#regressao)
5. [Hiperparâmetros e Controle de Overfitting](#hiper)
6. [Visualização de Árvores](#viz)
7. [Random Forest — Intuição](#rf_intro)
8. [Random Forest — Ajuste e Avaliação](#rf_fit)
9. [Importância de Variáveis](#importancia)
10. [Projeto Final — Random Forest Completo](#projeto)
11. [Apêndice Matemático Avançado](#apendice)

---

<a id="setup"></a>
# 0. Setup — bibliotecas e dados

Vamos usar o mesmo dataset do Módulo 8:
- `high_eval` como variável alvo (classificação)
- variáveis explicativas contínuas e categóricas

Isso permite comparar árvores com regressão logística.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, auc
)

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
ratings_df = pd.read_csv("/home/moacir/projects/ml/IBM/statistics/data/raw/teachingratings.csv")
ratings_df["high_eval"] = (ratings_df["eval"] >= 4.5).astype(int)

ratings_df.head()

<a id="intro"></a>
# 2. Introdução — Por que Árvores de Decisão?

Até agora, trabalhamos com modelos **lineares**:

- Regressão Linear  
- Regressão Logística  

Esses modelos assumem que:

$$
f(x) = \beta_0 + \beta_1 x_1 + \dots + \beta_k x_k
$$

Ou seja:

- relação linear entre variáveis  
- efeitos aditivos  

Mas o mundo real raramente é linear.

Exemplos:

- risco de crédito depende de faixas de renda  
- churn depende de combinações de comportamento  
- fraude depende de padrões não lineares  
- saúde depende de interações entre variáveis  

Árvores de decisão resolvem isso naturalmente.

---
# 2.1 O que é uma árvore de decisão?

Uma árvore é um modelo que:

- divide o espaço de dados em regiões  
- usando perguntas do tipo **sim/não**  
- até chegar a uma decisão final  

Exemplo:

```
O aluno estudou mais de 3 horas?
├── Sim → Ele passou?
└── Não → Ele passou?
```

Cada divisão é chamada de **nó**.  
Cada pergunta é chamada de **split**.  
Cada resposta final é uma **folha**.

---
# 2.2 Árvores capturam não linearidades automaticamente

Uma árvore pode aprender regras como:

- se beauty > 0.2 e students < 50 → alta avaliação  
- se age > 55 e beauty < -0.5 → baixa avaliação  

Isso é impossível para modelos lineares sem engenharia de features.

Árvores:

✔️ aprendem interações  
✔️ aprendem não linearidades  
✔️ aprendem regras complexas  
✔️ não exigem padronização  
✔️ funcionam bem com variáveis categóricas  

---
# 2.3 Exemplo visual simples

In [ ]:
from sklearn.tree import DecisionTreeClassifier

X = ratings_df[["beauty"]]
y = ratings_df["high_eval"]

tree_simple = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_simple.fit(X, y)

plt.figure(figsize=(12, 6))
plot_tree(tree_simple, feature_names=["beauty"], class_names=["0","1"], filled=True)
plt.title("Árvore de Decisão Simples (profundidade = 2)")
plt.show()

🟣 **Interpretação**

A árvore aprendeu regras do tipo:

- se beauty < x → baixa probabilidade  
- se beauty ≥ x → alta probabilidade  

Isso é uma **fronteira de decisão não linear**, mesmo com uma única variável.

---
# 2.4 Árvores são interpretáveis

Diferente de modelos lineares:

- você pode **ver** a árvore  
- você pode **ler** as regras  
- você pode **explicar** o modelo para qualquer pessoa  

Por isso árvores são muito usadas em:

- medicina  
- direito  
- finanças  
- auditoria  
- risco de crédito  

Onde interpretabilidade é essencial.

---
# 2.5 Árvores têm um problema: overfitting

Árvores profundas demais:

- aprendem ruído  
- memorizam o conjunto de treino  
- generalizam mal  

Exemplo:

- profundidade = 20 → quase sempre overfitting  
- profundidade = 3–6 → geralmente bom  

Por isso precisamos controlar:

- profundidade  
- número mínimo de amostras por nó  
- número mínimo de amostras por folha  

E mais tarde, usaremos **Random Forest** para resolver esse problema.

---
# 2.6 Conclusão desta parte

✔️ Árvores capturam não linearidades  
✔️ Árvores capturam interações  
✔️ Árvores são interpretáveis  
✔️ Árvores funcionam bem sem pré‑processamento  
✔️ Árvores podem sofrer overfitting  

Agora estamos prontos para:

**PARTE 3 — Árvores de Classificação (primeiro modelo completo).**

<a id="classificacao"></a>
# 3. Árvores de Classificação

Agora vamos construir nossa primeira **árvore de decisão para classificação**.

O objetivo é prever:

> **high_eval = 1 se eval ≥ 4.5**

usando múltiplas variáveis explicativas:

- beauty  
- age  
- students  
- female_dummy  

Vamos começar com uma árvore simples e depois aprofundar.

---
# 3.1 Preparando os dados

In [ ]:
X = ratings_df[["beauty", "age", "students", "female_dummy"]]
y = ratings_df["high_eval"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

len(X_train), len(X_test)

---
# 3.2 Ajustando uma árvore de classificação

Vamos começar com uma árvore **sem limitar profundidade**.

In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

In [ ]:
tree

---
# 3.3 Avaliando a árvore

In [ ]:
y_pred = tree.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

pd.DataFrame({
    "Acurácia": [acc],
    "Precisão": [prec],
    "Recall": [rec],
    "F1": [f1]
})

🟣 **Interpretação**

Árvores sem restrições tendem a:

- acertar muito no treino  
- errar mais no teste  

Isso é **overfitting**, e vamos ver isso claramente mais adiante.

---
# 3.4 Visualizando a árvore

In [ ]:
plt.figure(figsize=(18, 10))
plot_tree(
    tree,
    feature_names=X.columns,
    class_names=["0", "1"],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Árvore de Classificação — Sem Restrições")
plt.show()

🟣 **Interpretação**

- Cada **nó** contém uma pergunta (split)  
- Cada **folha** contém a classe prevista  
- Árvores profundas ficam difíceis de interpretar  

Isso reforça a necessidade de limitar a complexidade.

---
# 3.5 Controlando a profundidade

Vamos ajustar uma árvore com profundidade máxima = 3.

In [ ]:
tree3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree3.fit(X_train, y_train)

y_pred3 = tree3.predict(X_test)

pd.DataFrame({
    "Acurácia": [accuracy_score(y_test, y_pred3)],
    "Precisão": [precision_score(y_test, y_pred3)],
    "Recall": [recall_score(y_test, y_pred3)],
    "F1": [f1_score(y_test, y_pred3)]
})

🟣 **Interpretação**

- A árvore ficou mais simples  
- Geralmente generaliza melhor  
- Evita overfitting  

Árvores rasas são mais robustas.

---
# 3.6 Visualizando a árvore com profundidade 3

In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    tree3,
    feature_names=X.columns,
    class_names=["0", "1"],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Árvore de Classificação — Profundidade = 3")
plt.show()

🟣 **Interpretação**

Agora conseguimos **ler** a árvore:

- se beauty < x → baixa probabilidade  
- se students < y → alta probabilidade  
- se age > z → baixa probabilidade  

Árvores são extremamente interpretáveis.

---
# 3.7 Comparando árvore profunda vs árvore rasa

Vamos comparar desempenho no treino e teste.

In [ ]:
train_pred_deep = tree.predict(X_train)
train_pred_shallow = tree3.predict(X_train)

test_pred_deep = tree.predict(X_test)
test_pred_shallow = tree3.predict(X_test)

comparison = pd.DataFrame({
    "Modelo": ["Árvore Profunda", "Árvore Rasa"],
    "Acurácia Treino": [
        accuracy_score(y_train, train_pred_deep),
        accuracy_score(y_train, train_pred_shallow)
    ],
    "Acurácia Teste": [
        accuracy_score(y_test, test_pred_deep),
        accuracy_score(y_test, test_pred_shallow)
    ]
})

comparison

🟣 **Interpretação**

- A árvore profunda acerta quase tudo no treino → **overfitting**  
- A árvore rasa tem desempenho mais equilibrado → **melhor generalização**  

Isso é exatamente o que esperamos de árvores.

---
# 3.8 Conclusão desta parte

✔️ Construímos uma árvore de classificação  
✔️ Visualizamos a árvore  
✔️ Avaliamos desempenho  
✔️ Vimos overfitting na prática  
✔️ Aprendemos a controlar profundidade  

Agora estamos prontos para:

**PARTE 4 — Árvores de Regressão (versão contínua).**

<a id="regressao"></a>
# 4. Árvores de Regressão

Árvores de regressão são a versão contínua das árvores de classificação.

Em vez de prever:
- 0 ou 1

Elas prevêem:
- valores numéricos contínuos

Exemplo:
- preço de casa  
- tempo de entrega  
- nota de avaliação  
- risco financeiro  

Aqui vamos prever a variável **eval** (avaliação contínua do professor).

---
# 4.1 Preparando os dados

In [ ]:
X_reg = ratings_df[["beauty", "age", "students", "female_dummy"]]
y_reg = ratings_df["eval"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)

len(Xr_train), len(Xr_test)

---
# 4.2 Ajustando uma árvore de regressão

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(Xr_train, yr_train)

tree_reg

---
# 4.3 Avaliando a árvore de regressão

Usamos métricas de regressão:
- RMSE  
- MAE  
- R²  

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

yr_pred = tree_reg.predict(Xr_test)

rmse = mean_squared_error(yr_test, yr_pred, squared=False)
mae = mean_absolute_error(yr_test, yr_pred)
r2 = r2_score(yr_test, yr_pred)

pd.DataFrame({
    "RMSE": [rmse],
    "MAE": [mae],
    "R²": [r2]
})

🟣 **Interpretação**

Árvores de regressão tendem a:
- ter R² alto no treino  
- ter R² mais baixo no teste  

Isso indica **overfitting**, assim como nas árvores de classificação.

---
# 4.4 Visualizando a árvore de regressão

In [ ]:
plt.figure(figsize=(18, 10))
plot_tree(
    tree_reg,
    feature_names=X_reg.columns,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Árvore de Regressão — Sem Restrições")
plt.show()

🟣 **Interpretação**

Árvores de regressão criam **regiões constantes**:

- cada folha prevê um valor fixo  
- todas as observações naquela região recebem a mesma previsão  

Isso é muito diferente da regressão linear.

---
# 4.5 Controlando a profundidade

In [ ]:
tree_reg3 = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_reg3.fit(Xr_train, yr_train)

yr_pred3 = tree_reg3.predict(Xr_test)

pd.DataFrame({
    "RMSE": [mean_squared_error(yr_test, yr_pred3, squared=False)],
    "MAE": [mean_absolute_error(yr_test, yr_pred3)],
    "R²": [r2_score(yr_test, yr_pred3)]
})

🟣 **Interpretação**

- A árvore ficou mais simples  
- Generaliza melhor  
- Evita overfitting  

Árvores rasas são mais robustas também em regressão.

---
# 4.6 Visualizando a árvore com profundidade 3

In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    tree_reg3,
    feature_names=X_reg.columns,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Árvore de Regressão — Profundidade = 3")
plt.show()

---
# 4.7 Comparando árvore profunda vs árvore rasa

In [ ]:
train_pred_deep = tree_reg.predict(Xr_train)
train_pred_shallow = tree_reg3.predict(Xr_train)

test_pred_deep = tree_reg.predict(Xr_test)
test_pred_shallow = tree_reg3.predict(Xr_test)

comparison_reg = pd.DataFrame({
    "Modelo": ["Árvore Profunda", "Árvore Rasa"],
    "R² Treino": [
        r2_score(yr_train, train_pred_deep),
        r2_score(yr_train, train_pred_shallow)
    ],
    "R² Teste": [
        r2_score(yr_test, test_pred_deep),
        r2_score(yr_test, test_pred_shallow)
    ]
})

comparison_reg

🟣 **Interpretação**

- A árvore profunda memoriza o treino → **overfitting**  
- A árvore rasa generaliza melhor → **melhor modelo real**  

Isso é exatamente o comportamento esperado.

---
# 4.8 Conclusão desta parte

✔️ Árvores de regressão criam regiões constantes  
✔️ Capturam não linearidades automaticamente  
✔️ Sofrem overfitting quando profundas  
✔️ Árvores rasas generalizam melhor  
✔️ Funcionam bem sem pré‑processamento  

Agora estamos prontos para:

**PARTE 5 — Hiperparâmetros e Controle de Overfitting.**

<a id="hiper"></a>
# 5. Hiperparâmetros e Controle de Overfitting

Árvores de decisão são modelos extremamente flexíveis.

Essa flexibilidade é boa porque:
- capturam padrões complexos  
- aprendem interações  
- aprendem não linearidades  

Mas também é perigosa:
- árvores profundas demais **decoram** o conjunto de treino  
- isso leva a **overfitting**  

Por isso, controlar a complexidade da árvore é essencial.

---
# 5.1 Principais hiperparâmetros

Os hiperparâmetros mais importantes são:

### ✔️ `max_depth`
Profundidade máxima da árvore.

- valores altos → mais complexidade  
- valores baixos → modelo mais simples  

---

### ✔️ `min_samples_split`
Número mínimo de amostras para dividir um nó.

- valores altos → menos splits → árvore mais simples  

---

### ✔️ `min_samples_leaf`
Número mínimo de amostras em uma folha.

- evita folhas com 1 ou 2 observações  
- reduz overfitting  

---

### ✔️ `max_features`
Número de variáveis consideradas em cada split.

- importante para Random Forest  
- reduz correlação entre árvores  

---

Vamos ver esses hiperparâmetros na prática.

---
# 5.2 Comparando profundidades diferentes

In [ ]:
depths = range(1, 15)
train_scores = []
test_scores = []

for d in depths:
    model_d = DecisionTreeClassifier(max_depth=d, random_state=42)
    model_d.fit(X_train, y_train)
    
    train_scores.append(model_d.score(X_train, y_train))
    test_scores.append(model_d.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(depths, train_scores, label="Treino")
plt.plot(depths, test_scores, label="Teste")
plt.xlabel("Profundidade (max_depth)")
plt.ylabel("Acurácia")
plt.title("Impacto da Profundidade na Acurácia")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- profundidade baixa → underfitting  
- profundidade alta → overfitting  
- existe um ponto ótimo (bias–variance trade‑off)  

Esse gráfico é clássico em árvores de decisão.

---
# 5.3 Controlando min_samples_leaf

In [ ]:
leaf_sizes = range(1, 20)
train_scores_leaf = []
test_scores_leaf = []

for leaf in leaf_sizes:
    model_leaf = DecisionTreeClassifier(min_samples_leaf=leaf, random_state=42)
    model_leaf.fit(X_train, y_train)
    
    train_scores_leaf.append(model_leaf.score(X_train, y_train))
    test_scores_leaf.append(model_leaf.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(leaf_sizes, train_scores_leaf, label="Treino")
plt.plot(leaf_sizes, test_scores_leaf, label="Teste")
plt.xlabel("min_samples_leaf")
plt.ylabel("Acurácia")
plt.title("Impacto de min_samples_leaf na Acurácia")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- folhas muito pequenas → árvore memoriza o treino  
- folhas maiores → árvore mais suave e generaliza melhor  

`min_samples_leaf` é um dos hiperparâmetros mais importantes.

---
# 5.4 Controlando min_samples_split

In [ ]:
splits = range(2, 30)
train_scores_split = []
test_scores_split = []

for s in splits:
    model_split = DecisionTreeClassifier(min_samples_split=s, random_state=42)
    model_split.fit(X_train, y_train)
    
    train_scores_split.append(model_split.score(X_train, y_train))
    test_scores_split.append(model_split.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(splits, train_scores_split, label="Treino")
plt.plot(splits, test_scores_split, label="Teste")
plt.xlabel("min_samples_split")
plt.ylabel("Acurácia")
plt.title("Impacto de min_samples_split na Acurácia")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- valores baixos → árvore muito profunda  
- valores altos → árvore mais suave  

`min_samples_split` controla a agressividade dos splits.

---
# 5.5 Escolhendo a melhor árvore

Em geral:

- `max_depth` entre 3 e 6 funciona bem  
- `min_samples_leaf` entre 5 e 20 melhora generalização  
- `min_samples_split` entre 10 e 50 evita splits inúteis  

Mas o ideal é usar:

- validação cruzada  
- grid search  
- random search  

(isso será aprofundado no Módulo 10)

---
# 5.6 Conclusão desta parte

✔️ Árvores profundas sofrem overfitting  
✔️ Hiperparâmetros controlam a complexidade  
✔️ `max_depth` é o mais importante  
✔️ `min_samples_leaf` suaviza a árvore  
✔️ `min_samples_split` evita splits inúteis  
✔️ Gráficos ajudam a visualizar o bias–variance trade‑off  

Agora estamos prontos para:

**PARTE 6 — Visualização de Árvores (plot_tree e interpretação).**

<a id="viz"></a>
# 6. Visualização de Árvores de Decisão

Árvores de decisão são um dos modelos mais interpretáveis do Machine Learning.

Mas para aproveitar essa interpretabilidade, precisamos:

- visualizar a árvore  
- entender cada nó  
- interpretar cada split  
- ler as folhas corretamente  

Vamos aprender isso agora.

---
# 6.1 Ajustando uma árvore simples para visualização

In [ ]:
tree_viz = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)

tree_viz.fit(X_train, y_train)

---
# 6.2 Visualizando a árvore com plot_tree

In [ ]:
plt.figure(figsize=(18, 10))
plot_tree(
    tree_viz,
    feature_names=X.columns,
    class_names=["0", "1"],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Árvore de Classificação — Visualização Completa")
plt.show()

---
# 6.3 Como interpretar cada parte da árvore

Cada nó contém:

### ✔️ A regra (split)
Exemplo:
```
beauty <= 0.12
```

### ✔️ Gini impurity
Mede a pureza do nó:
- 0 → nó puro  
- 0.5 → mistura máxima (para 2 classes)  

Fórmula:
$$
G = 1 - p_1^2 - p_2^2
$$

### ✔️ samples
Quantas observações chegaram naquele nó.

### ✔️ value
Contagem de cada classe:
```
value = [n_class0, n_class1]
```

### ✔️ class
Classe prevista na folha.

---
# 6.4 Interpretando a árvore passo a passo

Vamos interpretar a árvore gerada:

- **Primeiro split**: a variável mais importante  
- **Nós à esquerda**: condição verdadeira  
- **Nós à direita**: condição falsa  

Exemplo:

```
beauty <= 0.12?
├── Sim → baixa avaliação
└── Não → alta avaliação
```

Árvores são literalmente **regras de decisão**.

---
# 6.5 Visualizando apenas a estrutura (sem detalhes)

In [ ]:
plt.figure(figsize=(18, 10))
plot_tree(
    tree_viz,
    feature_names=X.columns,
    class_names=["0", "1"],
    filled=False,
    rounded=True,
    fontsize=10
)
plt.title("Árvore — Estrutura Simples")
plt.show()

🟣 **Interpretação**

Essa visualização é útil quando:
- você quer ver apenas a estrutura  
- sem métricas  
- sem valores  

Ideal para apresentações.

---
# 6.6 Visualizando apenas as regras (sem gini, samples, value)

In [ ]:
plt.figure(figsize=(18, 10))
plot_tree(
    tree_viz,
    feature_names=X.columns,
    class_names=["0", "1"],
    filled=True,
    rounded=True,
    fontsize=10,
    impurity=False,
    proportion=True
)
plt.title("Árvore — Apenas Regras e Proporções")
plt.show()

🟣 **Interpretação**

- `impurity=False` remove o gini  
- `proportion=True` mostra proporções em vez de contagens  

Essa versão é ótima para explicar o modelo para não‑técnicos.

---
# 6.7 Exportando regras da árvore (interpretação textual)

Árvores podem ser convertidas em **regras if‑else**.

In [ ]:
from sklearn.tree import export_text

rules = export_text(tree_viz, feature_names=list(X.columns))
print(rules)

🟣 **Interpretação**

Agora temos o modelo como um conjunto de regras:

```
if beauty <= 0.12:
    if students <= 45:
        return 0
    else:
        return 1
else:
    return 1
```

Isso é extremamente útil para:
- auditoria  
- compliance  
- explicabilidade  
- documentação  
- relatórios executivos  

---
# 6.8 Conclusão desta parte

✔️ Aprendemos a visualizar árvores com plot_tree  
✔️ Interpretamos gini, samples, value e class  
✔️ Vimos como ler regras de decisão  
✔️ Geramos visualizações simplificadas  
✔️ Exportamos regras em formato textual  

Agora estamos prontos para:

**PARTE 7 — Random Forest: Intuição e Motivação.**

<a id="rf_intro"></a>
# 7. Random Forest — Intuição e Motivação

Até agora vimos:
- Árvores são interpretáveis  
- Árvores capturam não linearidades  
- Árvores capturam interações  

Mas também vimos um problema sério:

# **Árvores sofrem overfitting facilmente**

Uma árvore profunda:
- memoriza o conjunto de treino  
- generaliza mal  

Como resolver isso?

A resposta é:

# **Random Forest**

Um dos modelos mais poderosos e robustos do Machine Learning moderno.

---
# 7.1 A ideia central do Random Forest

Random Forest é baseado em uma ideia simples:

> **Combinar muitas árvores fracas → cria um modelo forte**

Isso é chamado de **ensemble** (comitê de modelos).

O processo:

1. Criamos **várias árvores**  
2. Cada árvore vê **um subconjunto aleatório dos dados**  
3. Cada árvore vê **um subconjunto aleatório das variáveis**  
4. As previsões são combinadas:
   - classificação → voto da maioria  
   - regressão → média  

O resultado:

✔️ menor variância  
✔️ menor overfitting  
✔️ melhor generalização  
✔️ mais robusto  

E ainda mantém boa interpretabilidade via **importância de variáveis**.

---
# 7.2 Por que funciona? (Intuição profunda)

Uma árvore individual tem:
- **alta variância**  
- **baixa viés**  

Random Forest reduz a variância porque:

- cada árvore é treinada em dados diferentes  
- cada árvore usa variáveis diferentes  
- erros individuais se cancelam  

É como perguntar a opinião de 200 especialistas diferentes:
- cada um vê parte da informação  
- cada um tem um viés diferente  
- mas a média das opiniões é muito mais estável  

Isso é o coração do Random Forest.

---
# 7.3 Bagging — o mecanismo por trás do Random Forest

Random Forest usa uma técnica chamada:

# **Bagging (Bootstrap Aggregating)**

Passos:

1. Sorteia amostras com reposição (bootstrap)  
2. Treina uma árvore em cada amostra  
3. Combina as previsões  

Isso reduz variância sem aumentar viés.

Random Forest adiciona mais uma aleatoriedade:

- em cada split, escolhe apenas **um subconjunto aleatório de variáveis**

Isso reduz correlação entre árvores → melhora desempenho.

---
# 7.4 Visualização conceitual (simulação simples)

In [ ]:
# Vamos simular 20 árvores rasas para mostrar a ideia de ensemble
trees = []
for i in range(20):
    t = DecisionTreeClassifier(
        max_depth=3,
        max_features=2,
        random_state=42 + i
    )
    t.fit(X_train, y_train)
    trees.append(t)

# Previsões individuais
preds = np.array([t.predict(X_test) for t in trees])

# Voto da maioria
final_pred = (preds.mean(axis=0) >= 0.5).astype(int)

accuracy_score(y_test, final_pred)

🟣 **Interpretação**

Mesmo com árvores rasas e simples:

- o ensemble melhora a estabilidade  
- reduz erros individuais  
- aumenta a acurácia  

Isso é exatamente o que o Random Forest faz — mas com dezenas ou centenas de árvores.

---
# 7.5 Vantagens do Random Forest

✔️ Captura não linearidades  
✔️ Captura interações automaticamente  
✔️ Resistente a outliers  
✔️ Funciona bem sem padronização  
✔️ Funciona bem com variáveis categóricas  
✔️ Reduz overfitting  
✔️ Excelente desempenho em dados tabulares  
✔️ Fácil de usar  

Por isso é um dos modelos mais usados em:
- risco de crédito  
- churn  
- fraude  
- saúde  
- marketing  
- indústria  
- ciência de dados aplicada  

---
# 7.6 Limitações do Random Forest

✔️ Menos interpretável que uma árvore única  
✔️ Pode ser lento com muitas árvores  
✔️ Pode consumir mais memória  
✔️ Não extrapola bem fora do intervalo dos dados  

Mas, na prática, é um dos melhores modelos para dados tabulares.

---
# 7.7 Conclusão desta parte

✔️ Árvores individuais sofrem overfitting  
✔️ Random Forest combina muitas árvores → modelo forte  
✔️ Bagging reduz variância  
✔️ Subamostragem de variáveis reduz correlação  
✔️ Ensemble melhora generalização  
✔️ Random Forest é robusto, poderoso e fácil de usar  

Agora estamos prontos para:

**PARTE 8 — Ajuste e Avaliação do Random Forest (modelo completo).**

<a id="rf_fit"></a>
# 8. Random Forest — Ajuste e Avaliação Completa

Agora que entendemos a intuição do Random Forest,
vamos ajustar um modelo completo para prever:

> **high_eval = 1 se eval ≥ 4.5**

Usando as variáveis:
- beauty  
- age  
- students  
- female_dummy  

E comparar com a árvore individual.

---
# 8.1 Preparando os dados

In [ ]:
X = ratings_df[["beauty", "age", "students", "female_dummy"]]
y = ratings_df["high_eval"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

---
# 8.2 Ajustando o Random Forest

Vamos começar com um modelo padrão:
- 200 árvores  
- profundidade livre  
- bootstrap ativado  
- max_features = sqrt(n_features) (padrão para classificação)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf

---
# 8.3 Avaliando o modelo

In [ ]:
y_pred_rf = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred_rf)
prec = precision_score(y_test, y_pred_rf)
rec = recall_score(y_test, y_pred_rf)
f1 = f1_score(y_test, y_pred_rf)

pd.DataFrame({
    "Acurácia": [acc],
    "Precisão": [prec],
    "Recall": [rec],
    "F1": [f1]
})

🟣 **Interpretação**

Random Forest geralmente:
- supera árvores individuais  
- tem melhor generalização  
- é mais robusto  

E isso aparece claramente nas métricas.

---
# 8.4 Comparando Random Forest vs Árvore Individual

In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)

comparison = pd.DataFrame({
    "Modelo": ["Árvore", "Random Forest"],
    "Acurácia": [
        accuracy_score(y_test, y_pred_tree),
        accuracy_score(y_test, y_pred_rf)
    ],
    "Precisão": [
        precision_score(y_test, y_pred_tree),
        precision_score(y_test, y_pred_rf)
    ],
    "Recall": [
        recall_score(y_test, y_pred_tree),
        recall_score(y_test, y_pred_rf)
    ],
    "F1": [
        f1_score(y_test, y_pred_tree),
        f1_score(y_test, y_pred_rf)
    ]
})

comparison

🟣 **Interpretação**

- A árvore individual sofre overfitting  
- O Random Forest reduz variância  
- O desempenho melhora em todas as métricas  

Isso é exatamente o comportamento esperado.

---
# 8.5 Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de Confusão — Random Forest")
plt.show()

---
# 8.6 Probabilidades e Curva ROC

In [ ]:
y_proba_rf = rf.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_proba_rf)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label="Random Forest")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aleatório")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Curva ROC — Random Forest")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
# 8.7 AUC — Área Sob a Curva

In [ ]:
auc_value = auc(fpr, tpr)
auc_value

🟣 **Interpretação**

- AUC alto indica excelente separação entre classes  
- Random Forest geralmente tem AUC superior a modelos lineares  

Isso confirma a força do modelo.

---
# 8.8 Overfitting: comparando treino vs teste

In [ ]:
train_pred_rf = rf.predict(X_train)

pd.DataFrame({
    "Conjunto": ["Treino", "Teste"],
    "Acurácia": [
        accuracy_score(y_train, train_pred_rf),
        accuracy_score(y_test, y_pred_rf)
    ]
})

🟣 **Interpretação**

- Random Forest ainda pode ter acurácia muito alta no treino  
- Mas generaliza muito melhor que uma árvore individual  

Overfitting é reduzido, não eliminado.

---
# 8.9 Conclusão desta parte

✔️ Ajustamos um Random Forest completo  
✔️ Avaliamos com métricas de classificação  
✔️ Comparação direta com árvore individual  
✔️ Matriz de confusão  
✔️ Curva ROC e AUC  
✔️ Análise de overfitting  

Agora estamos prontos para:

**PARTE 9 — Importância das Variáveis (Feature Importance).**

<a id="importancia"></a>
# 9. Importância das Variáveis no Random Forest

Uma das grandes vantagens do Random Forest é que ele fornece
uma medida clara e intuitiva de:

# **quais variáveis são mais importantes para o modelo**

Isso é extremamente útil para:
- interpretação  
- explicabilidade  
- seleção de variáveis  
- auditoria  
- relatórios executivos  

Vamos calcular e visualizar essas importâncias.

---
# 9.1 Ajustando novamente o Random Forest (para garantir consistência)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

---
# 9.2 Extraindo importâncias

In [ ]:
importances = rf.feature_importances_

importance_df = pd.DataFrame({
    "Variável": X.columns,
    "Importância": importances
}).sort_values("Importância", ascending=False)

importance_df

🟣 **Interpretação**

- Valores maiores → variáveis mais importantes  
- A soma das importâncias = 1  
- Importância é baseada na redução de impureza (Gini)  

Isso mostra **quais variáveis o modelo realmente usa**.

---
# 9.3 Visualização da importância das variáveis

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=importance_df,
    x="Importância",
    y="Variável",
    palette="viridis"
)
plt.title("Importância das Variáveis — Random Forest")
plt.xlabel("Importância")
plt.ylabel("Variável")
plt.show()

🟣 **Interpretação**

- A variável no topo é a mais importante  
- A escala é relativa (não absoluta)  
- Importâncias muito baixas podem indicar variáveis irrelevantes  

Isso ajuda a entender o comportamento do modelo.

---
# 9.4 Comparando com regressão logística

Na regressão logística:
- coeficientes representam efeitos lineares  
- odds ratio mede impacto multiplicativo  

No Random Forest:
- importâncias representam contribuição para reduzir impureza  
- não são efeitos lineares  

Portanto:

✔️ Importância ≠ coeficiente  
✔️ Importância ≠ causalidade  
✔️ Importância = contribuição para o modelo  

São conceitos diferentes, mas complementares.

---
# 9.5 Seleção de variáveis usando importâncias

Uma regra prática:

- variáveis com importância < 0.02 podem ser removidas  
- variáveis com importância alta devem ser mantidas  

Mas isso depende do contexto e do domínio.

In [ ]:
importance_df[importance_df["Importância"] < 0.02]

🟣 **Interpretação**

Essas variáveis têm impacto muito pequeno no modelo.

Em projetos reais, isso ajuda a:
- simplificar modelos  
- reduzir dimensionalidade  
- acelerar treinamento  
- melhorar interpretabilidade  

---
# 9.6 Visualização avançada: normalizando importâncias

In [ ]:
importance_df["Importância Normalizada"] = (
    importance_df["Importância"] / importance_df["Importância"].max()
)

importance_df

---
# 9.7 Conclusão desta parte

✔️ Random Forest fornece importâncias de variáveis  
✔️ Importância mede contribuição para reduzir impureza  
✔️ Visualização ajuda a interpretar o modelo  
✔️ Importância ≠ coeficiente (modelos lineares)  
✔️ Pode ser usada para seleção de variáveis  

Agora estamos prontos para:

**PARTE 10 — Projeto Final: Random Forest Completo.**

<a id="projeto"></a>
# 10. Projeto Final — Random Forest Completo

Neste projeto, vamos construir um pipeline completo de classificação usando:

- Árvores de Decisão  
- Random Forest  
- Avaliação completa  
- Curva ROC e AUC  
- Importância das Variáveis  

O objetivo é prever:

> **high_eval = 1 se eval ≥ 4.5**

Usando as variáveis:
- beauty  
- age  
- students  
- female_dummy  

Este projeto consolida todo o Módulo 9.

---
# 10.1 Preparação dos dados

In [ ]:
X = ratings_df[["beauty", "age", "students", "female_dummy"]]
y = ratings_df["high_eval"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

X_train.shape, X_test.shape

---
# 10.2 Ajustando uma Árvore de Classificação (baseline)

In [ ]:
tree = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=5,
    random_state=42
)

tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

baseline_results = pd.DataFrame({
    "Acurácia": [accuracy_score(y_test, y_pred_tree)],
    "Precisão": [precision_score(y_test, y_pred_tree)],
    "Recall": [recall_score(y_test, y_pred_tree)],
    "F1": [f1_score(y_test, y_pred_tree)]
})

baseline_results

---
# 10.3 Ajustando o Random Forest (modelo final)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rf_results = pd.DataFrame({
    "Acurácia": [accuracy_score(y_test, y_pred_rf)],
    "Precisão": [precision_score(y_test, y_pred_rf)],
    "Recall": [recall_score(y_test, y_pred_rf)],
    "F1": [f1_score(y_test, y_pred_rf)]
})

rf_results

---
# 10.4 Comparando Árvore vs Random Forest

In [ ]:
comparison = pd.concat(
    [baseline_results.assign(Modelo="Árvore"),
     rf_results.assign(Modelo="Random Forest")]
).set_index("Modelo")

comparison

🟣 **Interpretação**

- Random Forest supera a árvore individual em todas as métricas  
- Isso confirma a redução de variância e melhor generalização  
- O modelo final é mais robusto e estável  

---
# 10.5 Matriz de Confusão — Random Forest

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de Confusão — Random Forest")
plt.show()

---
# 10.6 Curva ROC e AUC

In [ ]:
y_proba_rf = rf.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_proba_rf)
auc_value = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {auc_value:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aleatório")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Curva ROC — Random Forest")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

auc_value

🟣 **Interpretação**

- AUC alto indica excelente separação entre classes  
- Random Forest captura padrões complexos que a árvore individual não captura  

---
# 10.7 Importância das Variáveis

In [ ]:
importance_df = pd.DataFrame({
    "Variável": X.columns,
    "Importância": rf.feature_importances_
}).sort_values("Importância", ascending=False)

importance_df

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=importance_df,
    x="Importância",
    y="Variável",
    palette="viridis"
)
plt.title("Importância das Variáveis — Random Forest")
plt.xlabel("Importância")
plt.ylabel("Variável")
plt.show()

🟣 **Interpretação**

- beauty é a variável mais importante  
- students e age também contribuem  
- female_dummy tem impacto menor  

Isso ajuda a explicar o modelo para stakeholders.

---
# 10.8 Overfitting: Treino vs Teste

In [ ]:
train_pred_rf = rf.predict(X_train)

pd.DataFrame({
    "Conjunto": ["Treino", "Teste"],
    "Acurácia": [
        accuracy_score(y_train, train_pred_rf),
        accuracy_score(y_test, y_pred_rf)
    ]
})

🟣 **Interpretação**

- A acurácia no treino é muito alta (quase 1.0)  
- A acurácia no teste é menor, mas ainda muito boa  
- Random Forest reduz overfitting, mas não elimina completamente  

---
# 10.9 Conclusão do Projeto Final

✔️ Construímos um pipeline completo com Random Forest  
✔️ Comparação com árvore individual  
✔️ Avaliação com acurácia, precisão, recall e F1  
✔️ Matriz de confusão  
✔️ Curva ROC e AUC  
✔️ Importância das variáveis  
✔️ Análise de overfitting  

Você agora domina **todo o ecossistema de Árvores e Random Forest**.

O próximo módulo aprofunda modelos ainda mais poderosos:

# **Módulo 10 — Gradient Boosting, XGBoost, LightGBM e CatBoost.**